In [1]:
from src.ppt_parser import extract_projects_from_pptx
from langchain.schema import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from dotenv import load_dotenv
import os

d:\DS_Journey\envs\medibot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import sys
print('Python path:', sys.executable)

Python path: d:\DS_Journey\envs\medibot\python.exe


In [4]:
!{sys.executable} -m pip install python-pptx langchain langchain-community langchain-core langchain-groq langchain-huggingface sentence-transformers faiss-cpu streamlit python-dotenv -q
print('All packages installed ✅')

All packages installed ✅


In [15]:
import os
import sys
sys.path.append('../')

from dotenv import load_dotenv
load_dotenv()

GROQ_API_KEY = os.getenv('GROQ_API_KEY')
os.environ['GROQ_API_KEY'] = GROQ_API_KEY

print('GROQ_API_KEY loaded:', 'Yes' if GROQ_API_KEY else 'Missing - check .env file')

GROQ_API_KEY loaded: Yes


In [9]:
from pptx import Presentation
import glob

# Check what pptx files are available
pptx_files = glob.glob('IntelliDecks/data/Dataset_project_repository.pptx')
print(f'Found {len(pptx_files)} pptx file(s):')
for f in pptx_files:
    print(' -', f)

Found 0 pptx file(s):


In [10]:
from pptx import Presentation
import glob

# Check what pptx files are available
pptx_files = glob.glob('IntelliDecks/data/Dataset_project_repository.pptx')
print(f'Found {len(pptx_files)} pptx file(s):')
for f in pptx_files:
    print(' -', f)

Found 0 pptx file(s):


In [11]:
import os
print(os.getcwd())

e:\GEN_AI\Project2\IntelliDecks


In [12]:
pptx_files = glob.glob('data/*.pptx')
print(pptx_files)

['data\\Dataset_project_repository.pptx']


In [ ]:
from pptx import Presentation

prs = Presentation("data/Dataset_project_repository.pptx")
print("Slides found:", len(prs.slides))

Slides found: 201


In [14]:
slide = prs.slides[0]

for shape in slide.shapes:
    if hasattr(shape, "text"):
        print(shape.text)


Project Repository
100 Real-World Projects  |  GenAI · ML/AI · Data Engineering
Sales & Delivery Intelligence Platform

100
Projects

25+
Domains

80+
Technologies


In [18]:
from pptx import Presentation
import glob

# Check what pptx files are available
pptx_files = glob.glob('data/Dataset_project_repository.pptx')
print(f'Found {len(pptx_files)} pptx file(s):')
for f in pptx_files:
    print(' -', f)

Found 1 pptx file(s):
 - data/Dataset_project_repository.pptx


In [20]:
# Load and inspect the presentation
prs = Presentation(pptx_files[0])
print(f'Total slides in presentation: {len(prs.slides)}')
print(f'Projects expected (2 slides each): {len(prs.slides) // 2}')

Total slides in presentation: 201
Projects expected (2 slides each): 100


In [21]:
# Preview first slide content
print('=== SLIDE 1 CONTENT PREVIEW ===')
slide1 = prs.slides[0]
for shape in slide1.shapes:
    if shape.has_text_frame:
        for para in shape.text_frame.paragraphs:
            if para.text.strip():
                print(para.text.strip())

=== SLIDE 1 CONTENT PREVIEW ===
Project Repository
100 Real-World Projects  |  GenAI · ML/AI · Data Engineering
Sales & Delivery Intelligence Platform
100
Projects
25+
Domains
80+
Technologies


In [43]:
from pptx import Presentation

def extract_projects_from_pptx(
        pptx_path,
        slides_per_project=2,
        skip_first_slide=True):

    prs = Presentation(pptx_path)

    slides = list(prs.slides)

    # Skip title/cover slide
    if skip_first_slide:
        slides = slides[1:]

    all_projects = []

    for i in range(0, len(slides), slides_per_project):

        slide_group = slides[i:i + slides_per_project]

        slide_numbers = [
            x + 2 if skip_first_slide else x + 1
            for x in range(i, i + len(slide_group))
        ]

        # Default values
        project_id = ""
        project_title = ""
        domain = ""
        tech_stack = ""
        team_size = ""
        duration = ""
        project_type = ""

        all_text_parts = []

        # Collect text from BOTH slides
        for slide in slide_group:

            for shape in slide.shapes:

                if not hasattr(shape, "text"):
                    continue

                text = shape.text.strip()

                if text:
                    all_text_parts.append(text)

        if not all_text_parts:
            continue

        # ------------------------------------------------------------------
        # FIXED PROJECT ID + TITLE
        # ------------------------------------------------------------------

        project_id = all_text_parts[0].strip()

        if len(all_text_parts) > 1:
            project_title = all_text_parts[1].strip()

        # ------------------------------------------------------------------
        # EXTRACT METADATA ONLY FROM FIRST PROJECT DETAILS BLOCK
        # ------------------------------------------------------------------

        project_details_idx = -1

        for idx, line in enumerate(all_text_parts):

            if line.strip() == "PROJECT DETAILS":
                project_details_idx = idx
                break

        if project_details_idx != -1:

            search_window = all_text_parts[
                project_details_idx:
                min(project_details_idx + 15, len(all_text_parts))
            ]

            for idx, line in enumerate(search_window):

                line = line.strip()

                if line == "Domain" and idx + 1 < len(search_window):
                    domain = search_window[idx + 1].strip()

                elif line == "Team Size" and idx + 1 < len(search_window):
                    team_size = search_window[idx + 1].strip()

                elif line == "Duration" and idx + 1 < len(search_window):
                    duration = search_window[idx + 1].strip()

                elif line == "Type" and idx + 1 < len(search_window):
                    project_type = search_window[idx + 1].strip()

                elif line == "TECHNOLOGY STACK" and idx + 1 < len(search_window):
                    tech_stack = search_window[idx + 1].strip()

        # ------------------------------------------------------------------
        # FULL TEXT FOR RAG
        # Includes BOTH slides:
        # - Problem Statement
        # - Solution Approach
        # - Expected Outcomes
        # - Technical Deep Dive
        # - Full Technology Breakdown
        # - Key Metrics & Business Impact
        # ------------------------------------------------------------------

        combined_text = "\n".join(all_text_parts)

        all_projects.append({
            "project_id": project_id,
            "project_title": project_title,
            "domain": domain,
            "tech_stack": tech_stack,
            "team_size": team_size,
            "duration": duration,
            "project_type": project_type,
            "slide_numbers": slide_numbers,
            "combined_text": combined_text
        })

    return all_projects

In [32]:
projects = extract_projects_from_pptx(
    "data/Dataset_project_repository.pptx"
)

print("Total projects:", len(projects))

print(projects[0]["slide_numbers"])
print(projects[-1]["slide_numbers"])

Total projects: 100
[2, 3]
[200, 201]


In [35]:
print("=== EXTRACTION SUMMARY ===")

print(f"Total Slides in PPT: {len(Presentation(pptx_files[0]).slides)}")
print(f"Total Projects Extracted: {len(projects)}")

print(f"First Project Slides : {projects[0]['slide_numbers']}")
print(f"Last Project Slides  : {projects[-1]['slide_numbers']}")

print("\n=== SAMPLE PROJECTS ===")

# Show first 3 projects only
for p in projects[:3]:
    print(f"\n{p['project_id']} | {p['project_title']}")
    print(f"  Domain: {p['domain']}")
    print(f"  Tech: {p['tech_stack']}")
    print(f"  Team: {p['team_size']} | Duration: {p['duration']}")
    print(f"  Slides: {p['slide_numbers']}")

=== EXTRACTION SUMMARY ===
Total Slides in PPT: 201
Total Projects Extracted: 100
First Project Slides : [2, 3]
Last Project Slides  : [200, 201]

=== SAMPLE PROJECTS ===

P001  ›  Customer Churn Prediction Engine  —  Technical Deep Dive | Customer Churn Prediction Engine
  Domain: PROBLEM CONTEXT
  Tech: Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL
  Team: 3 | Duration: Telecom
  Slides: [2, 3]

P002  ›  NLP-Powered Contract Review System  —  Technical Deep Dive | NLP-Powered Contract Review System
  Domain: PROBLEM CONTEXT
  Tech: Python  ·  BERT  ·  HuggingFace Transformers  ·  FastAPI  ·  React  ·  SharePoint
  Team: 2 | Duration: Legal
  Slides: [4, 5]

P003  ›  Real-Time Supply Chain Visibility Platform  —  Technical Deep Dive | Real-Time Supply Chain Visibility Platform
  Domain: PROBLEM CONTEXT
  Tech: Apache Kafka  ·  Databricks  ·  Delta Lake  ·  Python  ·  Power BI  ·  Azure
  Team: 5 | Duration: Manufacturing
  Slides: [6, 7]


In [36]:
print(projects[0]["combined_text"])

P001
Customer Churn Prediction Engine
ML/AI
PROBLEM STATEMENT
High customer attrition rate (18% annually) with no early-warning system, resulting in $12M annual revenue loss.
SOLUTION APPROACH
Built an ensemble ML model (XGBoost + LightGBM) trained on 3 years of usage data with real-time scoring via REST API. Deployed on AWS SageMaker with automated retraining pipeline.
PROJECT DETAILS
Domain
Telecom
Team Size
4 people
Duration
3 months
Type
ML/AI
TECHNOLOGY STACK
Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL
EXPECTED OUTCOMES
Reduced churn by 23%, saving ~$2.8M annually. Model achieves 87% recall on at-risk customers.
P001  ›  Customer Churn Prediction Engine  —  Technical Deep Dive
FULL TECHNOLOGY BREAKDOWN
Python
XGBoost
LightGBM
AWS SageMaker
Apache Kafka
PostgreSQL
KEY METRICS & BUSINESS IMPACT
Reduced churn by 23%, saving ~$2.8M annually. Model achieves 87% recall on at-risk customers.
4
Team Size
3
Duration
Telecom
Domain
PROBLEM CONTEXT
High 

In [37]:
projects = extract_projects_from_pptx(
    "data/Dataset_project_repository.pptx"
)

print("Total Projects:", len(projects))
print(projects[0])

Total Projects: 100
{'project_id': 'P001  ›  Customer Churn Prediction Engine  —  Technical Deep Dive', 'project_title': 'Customer Churn Prediction Engine', 'domain': 'PROBLEM CONTEXT', 'tech_stack': 'Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL', 'team_size': '3', 'duration': 'Telecom', 'project_type': 'ML/AI', 'slide_numbers': [2, 3], 'combined_text': 'P001\nCustomer Churn Prediction Engine\nML/AI\nPROBLEM STATEMENT\nHigh customer attrition rate (18% annually) with no early-warning system, resulting in $12M annual revenue loss.\nSOLUTION APPROACH\nBuilt an ensemble ML model (XGBoost + LightGBM) trained on 3 years of usage data with real-time scoring via REST API. Deployed on AWS SageMaker with automated retraining pipeline.\nPROJECT DETAILS\nDomain\nTelecom\nTeam Size\n4 people\nDuration\n3 months\nType\nML/AI\nTECHNOLOGY STACK\nPython  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL\nEXPECTED OUTCOMES\nReduced churn by 2

In [38]:
print(projects[0]["slide_numbers"])
print(projects[-1]["slide_numbers"])

[2, 3]
[200, 201]


In [39]:
for idx, line in enumerate(projects[0]["combined_text"].split("\n")):
    print(idx, "->", repr(line))

0 -> 'P001'
1 -> 'Customer Churn Prediction Engine'
2 -> 'ML/AI'
3 -> 'PROBLEM STATEMENT'
4 -> 'High customer attrition rate (18% annually) with no early-warning system, resulting in $12M annual revenue loss.'
5 -> 'SOLUTION APPROACH'
6 -> 'Built an ensemble ML model (XGBoost + LightGBM) trained on 3 years of usage data with real-time scoring via REST API. Deployed on AWS SageMaker with automated retraining pipeline.'
7 -> 'PROJECT DETAILS'
8 -> 'Domain'
9 -> 'Telecom'
10 -> 'Team Size'
11 -> '4 people'
12 -> 'Duration'
13 -> '3 months'
14 -> 'Type'
15 -> 'ML/AI'
16 -> 'TECHNOLOGY STACK'
17 -> 'Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL'
18 -> 'EXPECTED OUTCOMES'
19 -> 'Reduced churn by 23%, saving ~$2.8M annually. Model achieves 87% recall on at-risk customers.'
20 -> 'P001  ›  Customer Churn Prediction Engine  —  Technical Deep Dive'
21 -> 'FULL TECHNOLOGY BREAKDOWN'
22 -> 'Python'
23 -> 'XGBoost'
24 -> 'LightGBM'
25 -> 'AWS SageMaker'
26 -> 'Ap

In [42]:
projects = extract_projects_from_pptx(
    "data/Dataset_project_repository.pptx"
)

print(projects[0])

{'project_id': 'P001  ›  Customer Churn Prediction Engine  —  Technical Deep Dive', 'project_title': 'Customer Churn Prediction Engine', 'domain': 'PROBLEM CONTEXT', 'tech_stack': 'Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL', 'team_size': '3', 'duration': 'Telecom', 'project_type': 'ML/AI', 'slide_numbers': [2, 3], 'combined_text': 'P001\nCustomer Churn Prediction Engine\nML/AI\nPROBLEM STATEMENT\nHigh customer attrition rate (18% annually) with no early-warning system, resulting in $12M annual revenue loss.\nSOLUTION APPROACH\nBuilt an ensemble ML model (XGBoost + LightGBM) trained on 3 years of usage data with real-time scoring via REST API. Deployed on AWS SageMaker with automated retraining pipeline.\nPROJECT DETAILS\nDomain\nTelecom\nTeam Size\n4 people\nDuration\n3 months\nType\nML/AI\nTECHNOLOGY STACK\nPython  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL\nEXPECTED OUTCOMES\nReduced churn by 23%, saving ~$2.8M an

In [44]:
projects = extract_projects_from_pptx(
    "data/Dataset_project_repository.pptx"
)

print(projects[0])

{'project_id': 'P001', 'project_title': 'Customer Churn Prediction Engine', 'domain': 'Telecom', 'tech_stack': 'Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL', 'team_size': '4 people', 'duration': '3 months', 'project_type': 'ML/AI', 'slide_numbers': [2, 3], 'combined_text': 'P001\nCustomer Churn Prediction Engine\nML/AI\nPROBLEM STATEMENT\nHigh customer attrition rate (18% annually) with no early-warning system, resulting in $12M annual revenue loss.\nSOLUTION APPROACH\nBuilt an ensemble ML model (XGBoost + LightGBM) trained on 3 years of usage data with real-time scoring via REST API. Deployed on AWS SageMaker with automated retraining pipeline.\nPROJECT DETAILS\nDomain\nTelecom\nTeam Size\n4 people\nDuration\n3 months\nType\nML/AI\nTECHNOLOGY STACK\nPython  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL\nEXPECTED OUTCOMES\nReduced churn by 23%, saving ~$2.8M annually. Model achieves 87% recall on at-risk customers.\nP001

In [45]:
from pprint import pprint

pprint(projects[0])

{'combined_text': 'P001\n'
                  'Customer Churn Prediction Engine\n'
                  'ML/AI\n'
                  'PROBLEM STATEMENT\n'
                  'High customer attrition rate (18% annually) with no '
                  'early-warning system, resulting in $12M annual revenue '
                  'loss.\n'
                  'SOLUTION APPROACH\n'
                  'Built an ensemble ML model (XGBoost + LightGBM) trained on '
                  '3 years of usage data with real-time scoring via REST API. '
                  'Deployed on AWS SageMaker with automated retraining '
                  'pipeline.\n'
                  'PROJECT DETAILS\n'
                  'Domain\n'
                  'Telecom\n'
                  'Team Size\n'
                  '4 people\n'
                  'Duration\n'
                  '3 months\n'
                  'Type\n'
                  'ML/AI\n'
                  'TECHNOLOGY STACK\n'
                  'Python  ·  XGBoost  ·  LightGBM  

In [46]:
print("Total Projects:", len(projects))

Total Projects: 100


In [47]:
projects = extract_projects_from_pptx(pptx_files[0])

print("Total Projects:", len(projects))
print(projects[0])

Total Projects: 100
{'project_id': 'P001', 'project_title': 'Customer Churn Prediction Engine', 'domain': 'Telecom', 'tech_stack': 'Python  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL', 'team_size': '4 people', 'duration': '3 months', 'project_type': 'ML/AI', 'slide_numbers': [2, 3], 'combined_text': 'P001\nCustomer Churn Prediction Engine\nML/AI\nPROBLEM STATEMENT\nHigh customer attrition rate (18% annually) with no early-warning system, resulting in $12M annual revenue loss.\nSOLUTION APPROACH\nBuilt an ensemble ML model (XGBoost + LightGBM) trained on 3 years of usage data with real-time scoring via REST API. Deployed on AWS SageMaker with automated retraining pipeline.\nPROJECT DETAILS\nDomain\nTelecom\nTeam Size\n4 people\nDuration\n3 months\nType\nML/AI\nTECHNOLOGY STACK\nPython  ·  XGBoost  ·  LightGBM  ·  AWS SageMaker  ·  Apache Kafka  ·  PostgreSQL\nEXPECTED OUTCOMES\nReduced churn by 23%, saving ~$2.8M annually. Model achieves 87% recall on at-r

In [48]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

# Test embedding
test_vector = embeddings.embed_query('Customer Churn Prediction')
print(f'Embedding model loaded ✅')
print(f'Vector dimensions: {len(test_vector)}')

Embedding model loaded ✅
Vector dimensions: 384


In [50]:
from langchain.schema import Document
from langchain_community.vectorstores import FAISS

# Convert projects to LangChain Documents
docs = []
for p in projects:
    doc = Document(
        page_content=p['combined_text'],
        metadata={
            'project_id': p['project_id'],
            'project_title': p['project_title'],
            'domain': p['domain'],
            'project_type': p['project_type'],
            'tech_stack': p['tech_stack'],
            'team_size': p['team_size'],
            'duration': p['duration'],
            'slide_numbers': p['slide_numbers']
        }
    )
    docs.append(doc)

print(f'Created {len(docs)} LangChain documents')

# Build FAISS index
vectorstore = FAISS.from_documents(docs, embeddings)
vectorstore.save_local('../faiss_index')
print('FAISS index saved')

Created 100 LangChain documents
FAISS index saved


In [51]:
# Load and test retriever
vectorstore = FAISS.load_local(
    '../faiss_index',
    embeddings,
    allow_dangerous_deserialization=True
)
retriever = vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': 3})

query = 'Show me ML projects using Python'
retrieved = retriever.invoke(query)

print(f'Query: "{query}"')
print(f'Retrieved {len(retrieved)} projects:\n')
for doc in retrieved:
    m = doc.metadata
    print(f"  → {m['project_id']} | {m['project_title']} (Slides: {m['slide_numbers']})")

Query: "Show me ML projects using Python"
Retrieved 3 projects:

  → P043 | AI Tutor for Adaptive Learning (Slides: [86, 87])
  → P013 | Drug Discovery Target Identification (Slides: [26, 27])
  → P055 | ML Pipeline Orchestration Platform (Slides: [110, 111])


In [53]:
from langchain_groq import ChatGroq

llm = ChatGroq(model='llama-3.1-8b-instant', temperature=0.3)
print('Groq LLaMA3 model loaded')

Groq LLaMA3 model loaded


In [55]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

system_prompt = """
You are IntelliDecks, an AI assistant for project portfolio search.

Use ONLY the retrieved context.

For every answer include:
- Project ID
- Project Title
- Domain
- Project Type
- Tech Stack
- Team Size
- Duration
- Slide Numbers

If no relevant project exists, say:
'No matching projects found.'

{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', system_prompt),
    ('human', '{input}'),
])

question_answer_chain = create_stuff_documents_chain(llm, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)
print('RAG chain ready')

RAG chain ready


In [56]:
# Query 1 — By technology
response = rag_chain.invoke({'input': 'Show me all ML/AI projects'})
print('Query: Show me all ML/AI projects')
print('='*60)
print(response['answer'])

Query: Show me all ML/AI projects
Here are the ML/AI projects:

1. **P069** 
   - **Project ID**: P069
   - **Project Title**: Synthetic Data Generation for Model Training
   - **Domain**: FinTech / Healthcare
   - **Project Type**: ML/AI
   - **Tech Stack**: Python  ·  CTGAN  ·  SDV Library  ·  TensorFlow Privacy  ·  Great Expectations  ·  FastAPI
   - **Team Size**: 3 people
   - **Duration**: 10 weeks
   - **Slide Numbers**: Not specified

2. **P043** 
   - **Project ID**: P043
   - **Project Title**: AI Tutor for Adaptive Learning
   - **Domain**: EdTech
   - **Project Type**: NLP / GenAI
   - **Tech Stack**: Python  ·  PyTorch  ·  OpenAI API  ·  FastAPI  ·  React  ·  PostgreSQL
   - **Team Size**: 4 people
   - **Duration**: 4 months
   - **Slide Numbers**: Not specified

3. **P088** 
   - **Project ID**: P088
   - **Project Title**: AI-Driven Market Research Synthesis
   - **Domain**: Consulting / Market Research
   - **Project Type**: NLP / GenAI
   - **Tech Stack**: OpenAI GPT-

In [57]:
# Query 2 — By team size
response2 = rag_chain.invoke({'input': 'Show projects with team size less than 5'})
print('Query: Show projects with team size less than 5')
print('='*60)
print(response2['answer'])

Query: Show projects with team size less than 5
P057
Automated A/B Testing Platform
Data Engineering
Project Type
Data Engineering
Tech Stack
Python  ·  FastAPI  ·  PostgreSQL  ·  React  ·  Snowflake  ·  Airflow
Team Size
3 people
Duration
3 months
Slide Numbers
3

P058
Intelligent Procurement Analytics
Data Engineering
Project Type
Data Engineering
Tech Stack
Python  ·  Snowflake  ·  dbt  ·  Airflow  ·  Power BI  ·  News API
Team Size
3 people
Duration
3 months
Slide Numbers
3

P088
AI-Driven Market Research Synthesis
NLP / GenAI
Project Type
NLP / GenAI
Tech Stack
OpenAI GPT-4  ·  LangChain  ·  FAISS  ·  Python  ·  Streamlit  ·  PDF Parser
Team Size
2 people
Duration
5 weeks
Slide Numbers
2


In [58]:
# Query 3 — By domain
response3 = rag_chain.invoke({'input': 'Show me healthcare or pharma domain projects'})
print('Query: Show me healthcare or pharma domain projects')
print('='*60)
print(response3['answer'])

Query: Show me healthcare or pharma domain projects
P005
Project ID: P005
Project Title: GenAI Document Intelligence for Pharma
Domain: Pharma / Life Sciences
Project Type: NLP / GenAI
Tech Stack: OpenAI GPT-4  ·  LangChain  ·  FAISS  ·  Python  ·  Azure Blob Storage  ·  Streamlit
Team Size: 4 people
Duration: 3 months
Slide Numbers: 4

P063
Project ID: P063
Project Title: AI-Assisted Drug Interaction Checker
Domain: Pharma / Healthcare
Project Type: NLP / GenAI
Tech Stack: Python  ·  Neo4j  ·  OpenAI API  ·  FastAPI  ·  React  ·  PostgreSQL
Team Size: 3 people
Duration: 3 months
Slide Numbers: 3

P013
Project ID: P013
Project Title: Drug Discovery Target Identification
Domain: Pharma / Life Sciences
Project Type: ML/AI
Tech Stack: Python  ·  PyTorch Geometric  ·  BioBERT  ·  Neo4j  ·  RDKit  ·  AWS
Team Size: 6 people
Duration: 8 months
Slide Numbers: 6


In [59]:
# Query 4 — Similarity search
response4 = rag_chain.invoke({'input': 'Find projects similar to customer churn prediction'})
print('Query: Find projects similar to customer churn prediction')
print('='*60)
print(response4['answer'])

Query: Find projects similar to customer churn prediction
Project ID: P001
Project Title: Customer Churn Prediction Engine
Domain: Telecom
Project Type: ML/AI
Tech Stack: Python, XGBoost, LightGBM, AWS SageMaker, Apache Kafka, PostgreSQL
Team Size: 4 people
Duration: 3 months
Slide Numbers: 4

Project ID: P050
Project Title: Customer Lifetime Value Prediction
Domain: E-Commerce / Subscription
Project Type: ML/AI
Tech Stack: Python, Lifetimes library, Scikit-learn, Snowflake, dbt, Braze API
Team Size: 2 people
Duration: 6 weeks
Slide Numbers: 2

Project ID: P067
Project Title: Customer Segmentation & Personalization Engine
Domain: Retail / Banking
Project Type: ML/AI
Tech Stack: Python, UMAP, HDBSCAN, Scikit-learn, Snowflake, Braze
Team Size: 3 people
Duration: 2 months
Slide Numbers: 3


In [60]:
print(len(projects))
print(projects[0]["domain"])
print(projects[0]["team_size"])
print(projects[0]["duration"])
print(projects[0]["project_type"])

100
Telecom
4 people
3 months
ML/AI
